# Zephvion AI PIE — Customer Agent
**Project Intelligence Engine**: Dual-Lane RAG → Gemini Spec Generation → RAGAS Evaluation

Run every cell top-to-bottom. Only two API keys are required:
- `NOMIC_API_KEY` — from [atlas.nomic.ai](https://atlas.nomic.ai)
- `GEMINI_API_KEY` — from [aistudio.google.com](https://aistudio.google.com)


#Install dependencies

In [48]:
!pip install -q qdrant-client requests tqdm numpy

##Imports

In [49]:
import os, json, uuid, re, time
from pathlib import Path
from tqdm import tqdm
import numpy as np
import requests

from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance

print("Imports OK ✅")

Imports OK ✅


##API Keys

Replace with your real keys before running

In [91]:
os.environ["NOMIC_API_KEY"]  = "YOUR_API_KEY"   # <-- paste here
os.environ["GEMINI_API_KEY"] = "YOUR_API_KEY"  # <-- paste here

NOMIC_KEY  = os.environ["NOMIC_API_KEY"]
GEMINI_KEY = os.environ["GEMINI_API_KEY"]
print("Keys loaded ✅")

Keys loaded ✅


##Nomic Embed via HTTP API (no `nomic login` needed)

In [92]:
NOMIC_EMBED_URL = "https://api-atlas.nomic.ai/v1/embedding/text"
EMBED_MODEL     = "nomic-embed-text-v1.5"
EMBED_DIM       = 768

def embed_texts(texts: list[str], task_type="search_document") -> list[list[float]]:
    """Call Nomic Atlas HTTP API; returns list of embedding vectors."""
    headers = {
        "Authorization": f"Bearer {NOMIC_KEY}",
        "Content-Type": "application/json",
    }
    body = {
        "model": EMBED_MODEL,
        "texts": texts,
        "task_type": task_type,   # search_document | search_query | clustering
    }
    resp = requests.post(NOMIC_EMBED_URL, headers=headers, json=body, timeout=60)
    resp.raise_for_status()
    return resp.json()["embeddings"]

# Quick smoke-test
_test = embed_texts(["hello world"])
assert len(_test[0]) == EMBED_DIM, f"Unexpected dim: {len(_test[0])}"
print(f"Nomic embed OK ✅  dim={len(_test[0])}")

Nomic embed OK ✅  dim=768


##Synthetic Customer-Agent repository (ATOM chunks)

##In a real run: replace this with code that reads your repo zip.
##Each chunk is one atomic logic unit (function / class / decision rule).

In [93]:
CODE_CHUNKS = [
    {
        "id": "CODE-001",
        "lane": "code",
        "file": "marketplace/plugin_loader.py",
        "text": (
            "def load_plugin(plugin_id: str, version: str) -> Plugin:\n"
            "    \"\"\"Dynamically loads a marketplace plugin by ID and semver version."
            " Validates manifest.json before execution.\"\"\"\n"
            "    manifest = fetch_manifest(plugin_id, version)\n"
            "    validate_manifest_schema(manifest)\n"
            "    sandbox = SandboxEnvironment(resource_limits=DEFAULT_LIMITS)\n"
            "    return sandbox.load(manifest)"
        ),
    },
    {
        "id": "CODE-002",
        "lane": "code",
        "file": "marketplace/plugin_loader.py",
        "text": (
            "def validate_manifest_schema(manifest: dict) -> None:\n"
            "    required_keys = ['plugin_id','version','runtime','permissions','checksum']\n"
            "    for k in required_keys:\n"
            "        if k not in manifest:\n"
            "            raise ManifestValidationError(f'Missing key: {k}')\n"
            "    if manifest['runtime'] not in ALLOWED_RUNTIMES:\n"
            "        raise RuntimeNotSupportedError(manifest['runtime'])"
        ),
    },
    {
        "id": "CODE-003",
        "lane": "code",
        "file": "marketplace/security.py",
        "text": (
            "class SkillSecurityScanner:\n"
            "    \"\"\"Stub for future security scanning of submitted skills.\"\"\"\n"
            "    # TODO: implement static analysis, CVE checks, permission audits\n"
            "    def scan(self, plugin_path: str) -> ScanReport:\n"
            "        raise NotImplementedError('Scanner not yet implemented')"
        ),
    },
    {
        "id": "CODE-004",
        "lane": "code",
        "file": "marketplace/versioning.py",
        "text": (
            "def is_semver_compatible(required: str, available: str) -> bool:\n"
            "    \"\"\"Returns True if 'available' satisfies the semver range 'required'.\"\"\"\n"
            "    req = semver.VersionInfo.parse(required)\n"
            "    avail = semver.VersionInfo.parse(available)\n"
            "    return avail >= req and avail.major == req.major"
        ),
    },
    {
        "id": "CODE-005",
        "lane": "code",
        "file": "marketplace/sandbox.py",
        "text": (
            "class SandboxEnvironment:\n"
            "    DEFAULT_LIMITS = {'cpu_ms': 2000, 'mem_mb': 256, 'network': False}\n"
            "    def __init__(self, resource_limits: dict):\n"
            "        self.limits = resource_limits\n"
            "    def load(self, manifest):\n"
            "        # Runs plugin inside a gVisor/WASM sandbox\n"
            "        return execute_in_sandbox(manifest, self.limits)"
        ),
    },
    {
        "id": "CODE-006",
        "lane": "code",
        "file": "marketplace/submission_api.py",
        "text": (
            "@app.post('/skills/submit')\n"
            "async def submit_skill(payload: SkillSubmission):\n"
            "    \"\"\"Endpoint: developer uploads a new skill package.\"\"\"\n"
            "    store_artifact(payload)\n"
            "    queue_review(payload.plugin_id)  # manual review queue\n"
            "    return {'status': 'pending_review', 'plugin_id': payload.plugin_id}"
        ),
    },
    {
        "id": "CODE-007",
        "lane": "code",
        "file": "marketplace/submission_api.py",
        "text": (
            "def queue_review(plugin_id: str):\n"
            "    \"\"\"Adds plugin to human-review queue. No automated security check yet.\"\"\"\n"
            "    ReviewQueue.enqueue({'plugin_id': plugin_id, 'ts': utcnow()})"
        ),
    },
    {
        "id": "CODE-008",
        "lane": "code",
        "file": "marketplace/permissions.py",
        "text": (
            "PERMISSION_REGISTRY = {\n"
            "    'read_crm': 'Read customer CRM records',\n"
            "    'write_crm': 'Write/update CRM records',\n"
            "    'send_email': 'Send emails via platform',\n"
            "    'external_api': 'Call external HTTP endpoints',\n"
            "}\n"
            "# Plugins must declare required permissions in manifest.json"
        ),
    },
]

print(f"Code chunks ready: {len(CODE_CHUNKS)} ✅")

Code chunks ready: 8 ✅


##Decision Lane data (jira_decisions_customer_agent.json)

In [94]:
DECISION_DATA = [
    {
        "id": "DECISION-016",
        "title": "Move to Pluggable Plugin Architecture",
        "description": (
            "System transitions from monolithic add-ons to a modular Pluggable Plugin "
            "Architecture. Each plugin runs inside a sandboxed execution environment "
            "(gVisor or WASM) with explicit versioning via semver. Plugins declare "
            "their permissions and runtime dependencies in a manifest.json. "
            "Rollback is supported per plugin version without full system redeploy."
        ),
        "rationale": (
            "Monolithic builds caused cascading failures on updates and blocked "
            "independent team delivery. The new architecture allows teams to ship "
            "plugins independently with sandboxed isolation preventing cross-plugin "
            "contamination."
        ),
        "impact": "High — affects all marketplace skill submissions and runtime execution",
        "status": "Accepted",
        "date": "2024-11-03",
    },
    {
        "id": "DECISION-021",
        "title": "Standardized Skill Metadata Schema",
        "description": (
            "Every skill published to the marketplace must include a manifest.json "
            "declaring: plugin_id (UUID), version (semver), runtime (python3.11|node18), "
            "permissions (allowlist), dependencies (name+version), and a SHA-256 checksum."
        ),
        "rationale": "Improves validation, reproducibility, and automated security scanning.",
        "impact": "Medium — breaking change for existing skill developers",
        "status": "Accepted",
        "date": "2024-11-18",
    },
    {
        "id": "DECISION-028",
        "title": "Security Review Gate Before Marketplace Publication",
        "description": (
            "New submissions must pass an automated security scan (static analysis + "
            "CVE dependency check) AND a human approval before being listed. "
            "Scanner output must be stored with the plugin artifact for auditability."
        ),
        "rationale": "Prevents malicious plugins from reaching production customers.",
        "impact": "High — adds latency to submission pipeline; requires scanner service",
        "status": "Proposed",
        "date": "2025-01-07",
    },
]

with open("jira_decisions_customer_agent.json", "w") as f:
    json.dump(DECISION_DATA, f, indent=2)

# Build decision chunks for Lane 2
DECISION_CHUNKS = [
    {
        "id": d["id"],
        "lane": "decision",
        "text": f"{d['id']} — {d['title']}\n{d['description']}\nRationale: {d.get('rationale','')}",
    }
    for d in DECISION_DATA
]

print(f"Decision file saved ✅  Decision chunks: {len(DECISION_CHUNKS)}")

Decision file saved ✅  Decision chunks: 3


##Embed both lanes with Nomic

In [95]:
ALL_CHUNKS = CODE_CHUNKS + DECISION_CHUNKS

BATCH_SIZE = 16  # Nomic free tier supports up to 32 texts per request
all_embeddings = []

texts = [c["text"] for c in ALL_CHUNKS]

for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding"):
    batch = texts[i : i + BATCH_SIZE]
    vecs  = embed_texts(batch, task_type="search_document")
    all_embeddings.extend(vecs)
    time.sleep(0.3)  # gentle rate-limit buffer

assert len(all_embeddings) == len(ALL_CHUNKS)
print(f"Embeddings ready ✅  total={len(all_embeddings)}  dim={len(all_embeddings[0])}")

Embedding: 100%|██████████| 1/1 [00:01<00:00,  1.02s/it]

Embeddings ready ✅  total=11  dim=768


##Qdrant in-memory vector store

In [96]:
client = QdrantClient(":memory:")

# create_collection replaces deprecated recreate_collection
client.create_collection(
    collection_name="ai_pie",
    vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
)

points = [
    PointStruct(
        id=i,
        vector=all_embeddings[i],
        payload=ALL_CHUNKS[i],
    )
    for i in range(len(ALL_CHUNKS))
]

client.upsert(collection_name="ai_pie", points=points)
print(f"Qdrant indexed {len(points)} points ✅")

Qdrant indexed 11 points ✅


##Dual-lane retrieval

In [97]:
def retrieve(query, top_k=6):
    q_vec = embed_texts([query], task_type="search_query")[0]

    hits = client.query_points(
        collection_name="ai_pie",
        query=q_vec,
        limit=top_k
    ).points

    code_hits = [h for h in hits if h.payload["lane"] == "code"]
    decision_hits = [h for h in hits if h.payload["lane"] == "decision"]

    return code_hits, decision_hits

print("Retrieval function ready ✅")

Retrieval function ready ✅


##Gemini spec generator

In [98]:
GEMINI_URL = (
    "https://generativelanguage.googleapis.com/v1beta/models/"
    f"gemini-2.5-flash:generateContent?key={GEMINI_KEY}"
)

# Initialize Google Generative AI model
import google.generativeai as genai
genai.configure(api_key=GEMINI_KEY)
model = genai.GenerativeModel('gemini-2.5-flash')


SYSTEM_PROMPT = """\
You are the AI PIE (Project Intelligence Engine) for Zephvion's Customer Agent project.
You generate structured technical specifications from raw feature requests.
You MUST output ONLY valid JSON — no markdown fences, no explanation, no preamble.
The JSON must have exactly these top-level keys:
  bdd_intent_contract, completeness_flags, suggested_kpis,
  project_specific_edge_cases, three_point_risk_report
"""

def build_prompt(feature_request: str, code_hits, decision_hits) -> str:
    code_ctx = "\n\n".join(
        f"[{h.payload.get('file','?')}]\n{h.payload['text'][:400]}"
        for h in code_hits
    )
    dec_ctx = "\n\n".join(
        h.payload["text"][:500] for h in decision_hits
    )
    return f"""\
Feature Request:
{feature_request}

=== CODE CONTEXT (Lane 1) ===
{code_ctx}

=== DECISION CONTEXT (Lane 2) ===
{dec_ctx}

STRICT RULES:
- Use ONLY the provided context
- Do NOT hallucinate
- Every statement must be grounded in the context
- If missing, say "Not specified in context"

You must produce:

1) JSON output (same schema as before)
2) A section called "grounded_explanations" with 5-10 clear factual sentences

UPDATED SCHEMA:
{{
  "bdd_intent_contract": {{
    "given": "...",
    "when": "...",
    "then": "..."
  }},
  "completeness_flags": {{
    "missing_requirements": [],
    "missing_nonfunctional": []
  }},
  "suggested_kpis": [],
  "project_specific_edge_cases": [],
  "three_point_risk_report": [],

  "grounded_explanations": [
    "The system uses ...",
    "The function ... handles ...",
    "The API ... processes ..."
  ]
}}

Output ONLY valid JSON.
"""


def clean_json_response(raw: str) -> str:
    """Strip markdown code fences if Gemini wraps output in ```json ... ```."""
    raw = raw.strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)
    return raw.strip()


def generate_spec(query, code_hits, decision_hits):

    code_context = "\n\n".join([h.payload["text"] for h in code_hits])
    decision_context = "\n\n".join([h.payload["text"] for h in decision_hits])

    prompt = f"""
You are generating a STRICTLY GROUNDED technical specification.

CRITICAL RULES:
- Use ONLY the provided context
- Do NOT hallucinate
- Every item MUST be a factual statement from context
- NO empty fields allowed

You MUST fill ALL fields with REAL content.

OUTPUT JSON:

{{
  "bdd_intent_contract": {{
    "given": "The system operates on {query}",
    "when": "A user triggers the feature",
    "then": "The system processes based on retrieved logic"
  }},
  "completeness_flags": {{
    "missing_requirements": ["Not specified in context"],
    "missing_nonfunctional": ["Not specified in context"]
  }},
  "suggested_kpis": [
    "Processing latency based on system flow",
    "Error rate observed in validation logic"
  ],
  "project_specific_edge_cases": [
    "The system handles invalid inputs based on validation logic",
    "The system processes incomplete data using fallback mechanisms",
    "The system ensures safe execution when unexpected inputs occur"
  ],
  "three_point_risk_report": [
    "Risk of invalid input causing failures in processing pipeline",
    "Risk of missing validation leading to incorrect outputs",
    "Risk of performance degradation under high load"
  ]
}}

CONTEXT:
{code_context}

{decision_context}

USER REQUEST:
{query}
"""

    response = model.generate_content(prompt)

    raw = response.text.strip()

    import re, json
    raw = re.sub(r"^```(?:json)?", "", raw)
    raw = re.sub(r"```$", "", raw).strip()

    try:
        spec = json.loads(raw)
    except:
        print("⚠️ JSON parse failed, returning fallback")
        spec = {}

    return spec


print("Spec generator ready ✅")

Spec generator ready ✅


##Run the pipeline

In [100]:
FEATURE_REQUEST = "Implement an automated security scanner for new skill submissions."

t0 = time.time()

code_hits, decision_hits = retrieve(FEATURE_REQUEST, top_k=6)

spec = generate_spec(FEATURE_REQUEST, code_hits, decision_hits)

evaluate(spec)

elapsed = time.time() - t0
print(f"\n⏱  Hot-query time: {elapsed:.1f}s  (target < 30s)")

if "error" in spec:
    print("\n⚠️  JSON parse failed — raw output below:")
    print(spec.get("raw_response", ""))
else:
    print("\n✅ Spec generated successfully")
    print(json.dumps(spec, indent=2))

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 8172.60ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 8365.68ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1315.87ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 20555.05ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 4375.85ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 9686.40ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3312.77ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encodi

TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 23.473565928s.

##RAGAS-style evaluation (F/P/R)

In [88]:
import re
import json

def tokenize(text: str) -> set:
    return set(re.sub(r"[^a-z0-9 ]", " ", text.lower()).split())


def overlap_score(a: str, b: str) -> float:
    ta, tb = tokenize(a), tokenize(b)
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)  # Jaccard similarity


def evaluate_spec(spec: dict, code_hits, decision_hits) -> dict:
    all_hits = code_hits + decision_hits
    context_texts = [h.payload["text"] for h in all_hits]
    context_blob  = " ".join(context_texts)

    # Faithfulness: are edge cases grounded in retrieved context?
    edge_cases = [
        ec["case"] if isinstance(ec, dict) else str(ec)
        for ec in spec.get("project_specific_edge_cases", [])
    ]
    risk_items = [
        r["description"] if isinstance(r, dict) else str(r)
        for r in spec.get("three_point_risk_report", [])
    ]
    claims = edge_cases + risk_items

    faithful_count = sum(
        1 for c in claims
        if any(overlap_score(c, ctx) > 0.08 for ctx in context_texts)
    )
    faithfulness = faithful_count / max(len(claims), 1)

    # Context Recall: key decision keywords appear in spec?
    decision_keywords = [
        "sandbox", "versioning", "semver", "manifest", "permission",
        "plugin", "monolithic", "checksum", "runtime", "isolation",
        "pluggable", "security", "scanner", "static analysis",
    ]
    spec_blob = json.dumps(spec).lower() # spec is already a dict at this point due to the fix below
    recalled  = sum(1 for kw in decision_keywords if kw in spec_blob)
    recall    = recalled / len(decision_keywords)

    # --- Precision: fraction of spec KPIs that are measurable/concrete ---
    kpis = spec.get("suggested_kpis", [])
    concrete_kpi_count = sum(
        1 for k in kpis
        if isinstance(k, dict) and k.get("target", "") not in ("", "TBD", "N/A")
    )
    precision = concrete_kpi_count / max(len(kpis), 1)

    # --- FPR composite score ---
    fpr = (faithfulness + precision + recall) / 3

    return {
        "faithfulness":    round(faithfulness, 6),
        "precision":       round(precision, 6),
        "recall":          round(recall, 6),
        "fpr_composite":   round(fpr, 6),
        "pass":            fpr > 0.60,
        "faithful_claims": faithful_count,
        "total_claims":    len(claims),
        "recalled_kws":    recalled,
        "total_kws":       len(decision_keywords),
    }

# Fix: Ensure 'spec' is a dictionary before passing it to evaluate_spec.
# The 'generate_spec' function returns a string, but 'evaluate_spec' expects a dict.
# Since we cannot change 'generate_spec' in a prior cell, we create a dummy dict here.
if not isinstance(spec, dict):
    print("Warning: 'spec' is not a dictionary. Evaluation metrics will be based on a dummy spec.")
    # Create a dummy spec that evaluate_spec can process without AttributeError
    spec = {
        "project_specific_edge_cases": [],
        "three_point_risk_report": [],
        "suggested_kpis": [],
        # Add other keys if 'evaluate_spec' expects them to avoid future errors
        "bdd_intent_contract": {"given": "", "when": "", "then": ""},
        "completeness_flags": {"missing_requirements": [], "missing_nonfunctional": []},
        "grounded_explanations": []
    }

metrics = evaluate_spec(spec, code_hits, decision_hits)

print("\n📊 RAGAS-style Evaluation Report")
print("=" * 40)
for k, v in metrics.items():
    print(f"  {k:<22}: {v}")
print("=" * 40)
status = "✅ PASS" if metrics["pass"] else "❌ FAIL"
print(f"  FPR > 0.60 threshold: {status}")


📊 RAGAS-style Evaluation Report
  faithfulness          : 1.0
  precision             : 0.0
  recall                : 0.6429
  fpr_composite         : 0.5476
  pass                  : False
  faithful_claims       : 9
  total_claims          : 9
  recalled_kws          : 9
  total_kws             : 14
  FPR > 0.60 threshold: ❌ FAIL


##Save deliverables

In [62]:
with open("sample_output.json", "w") as f:
    json.dump(spec, f, indent=2)

evaluation_report = {
    "feature_request": FEATURE_REQUEST,
    "hot_query_seconds": round(elapsed, 2),
    "ragas_metrics": metrics,
    "retrieved_chunks": {
        "code": [h.payload.get("id", h.payload.get("file")) for h in code_hits],
        "decision": [h.payload.get("id") for h in decision_hits],
    },
}

with open("evaluation.json", "w") as f:
    json.dump(evaluation_report, f, indent=2)

print("✅ sample_output.json saved")
print("✅ evaluation.json saved")

# If running in Google Colab, download files automatically
try:
    from google.colab import files
    files.download("sample_output.json")
    files.download("evaluation.json")
    files.download("jira_decisions_customer_agent.json")
except ImportError:
    pass  # not in Colab — files are in the working directory

✅ sample_output.json saved
✅ evaluation.json saved


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Summary

| Component | Implementation |
|-----------|----------------|
| **Lane 1 — Code** | 8 ATOM chunks from Customer Agent repo (plugin_loader, security, versioning, sandbox, submission_api, permissions) |
| **Lane 2 — Decision** | DECISION-016, DECISION-021, DECISION-028 from Jira JSON |
| **Embeddings** | Nomic `nomic-embed-text-v1.5` via HTTP API, dim=768 |
| **Vector DB** | Qdrant in-memory, Cosine similarity |
| **Generator** | Google Gemini 1.5 Flash with structured JSON prompt |
| **Evaluation** | Custom RAGAS-style F/P/R scorer (faithfulness + precision + recall) |
| **Deliverables** | `sample_output.json`, `evaluation.json`, `jira_decisions_customer_agent.json` |
